In [0]:
import mlflow
import mlflow.spark
from pyspark.sql import functions as F

# Charger le modèle déjà enregistré, sans réentraînement
mlflow.set_registry_uri("databricks-uc")

model_uri = (
    "models:/workspace.5lake_lab.random_forest_tip_model/1"
)

loaded_model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/workspace/5lake_lab/raw/mlflow_tmp"
)

# Lire la table mise à jour par ton pipeline
silver_df = spark.table(
    "workspace.5lake_lab.silver_trips_pipeline"
)

scoring_df = (
    silver_df
    .withColumn(
        "pickup_hour",
        F.hour("tpep_pickup_datetime")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("tpep_pickup_datetime")
    )
    .dropna(subset=[
        "trip_distance",
        "trip_duration_min",
        "pickup_hour",
        "day_of_week"
    ])
)

# Produire les prédictions
predictions = (
    loaded_model.transform(scoring_df)
    .withColumn("model_version", F.lit(1))
    .drop("features")
)

# Actualiser la table à chaque exécution
(
    predictions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.5lake_lab.gold_predictions")
)

display(
    spark.table("workspace.5lake_lab.gold_predictions")
    .select("tip_pct", "prediction", "model_version")
    .limit(10)
)

tip_pct,prediction,model_version
24.14,15.658809270604758,1
23.87,15.859316420441255,1
12.14,15.116699010587727,1
11.61,14.701069473688545,1
2.58,15.178499032154015,1
8.27,15.33252224743907,1
27.31,14.7361014731314,1
2.68,14.830835119704423,1
15.74,15.123776349269354,1
22.34,15.370155351423337,1
